# Master Backtest — Combined 4-Strategy Portfolio

Combines pre-computed daily P&L from four strategies into a single vol-targeted portfolio:

| Strategy | Notebook | Output file | Unit |
|----------|----------|-------------|------|
| **TSMOM** | `01_momentum_research_*.ipynb` | `outputs/tsmom_portfolio_equity_local_*.csv` | decimal return |
| **Lease Rate Curve** | `silver_gold_lease_curve_v2.ipynb` | `outputs/06_portfolio_pnl.csv` | bp on notional |
| **COMEX EFP** | `comex_efp_expansion.ipynb` | `outputs_comex/06_combined_portfolio.csv` | bp on notional |
| **Cash & Carry** | `comex_efp_expansion.ipynb` | `outputs_comex/cc_trades_gc/si.csv` | bp (trade log) |

> **Prerequisite**: Run all four strategy notebooks first to ensure CSV outputs are current.

**Methodology**: Each strategy is independently EWMA vol-targeted to 10% annualised, capital-weighted (60/10/20/10), then a portfolio-level vol overlay re-scales to 10%.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import glob, warnings

# ── CAPITAL ALLOCATION ───────────────────────────────────────────────────────
NAV_USD         = 100_000_000          # $100M reference book
CAPITAL_WEIGHTS = {'tsmom': 0.60, 'carry': 0.10, 'lease': 0.20, 'efp': 0.10}
RISK_BUDGETS    = {'tsmom': 0.30, 'carry': 0.10, 'lease': 0.20, 'efp': 0.20}

# ── VOL TARGETING ──────────────────────────────────────────────────────
TARGET_STRAT_VOL = 0.10   # per-strategy annualised vol target before weighting
TARGET_PORT_VOL  = 0.10   # combined portfolio vol target
EWMA_LAMBDA      = 0.94
LEV_CAP          = 2.0    # per-strategy max leverage scalar
PORT_LEV_CAP     = 2.0    # portfolio-level max leverage scalar
ANN              = 252

# ── BACKTEST PERIODS ────────────────────────────────────────────────
IS_END    = pd.Timestamp('2022-12-31')
OOS_START = pd.Timestamp('2023-01-01')
FULL_START = pd.Timestamp('2018-01-01')
STRESS_PERIODS = {
    'COVID (2020-03)':       ('2020-03-01', '2020-06-30'),
    'Russia/Ukraine (2022)': ('2022-02-24', '2022-06-30'),
    'SVB (2023-03)':         ('2023-03-01', '2023-05-31'),
    'Tariff Shock (2025)':   ('2025-03-01', '2025-04-30'),
}
PAUSE_DD_THRESHOLD = 0.10   # flag if portfolio drawdown > 10%
HARD_CAP_DD        = 0.18   # breach = system pause

# ── PATHS ────────────────────────────────────────────────────────────
OUT_DIR   = Path('outputs')
COMEX_DIR = Path('outputs_comex')
MB_DIR    = Path('outputs_masterbacktest')
MB_DIR.mkdir(exist_ok=True)

print(f'NAV reference: ${NAV_USD/1e6:.0f}M')
print(f'Capital weights: TSMOM={CAPITAL_WEIGHTS["tsmom"]:.0%}  '
      f'C&C={CAPITAL_WEIGHTS["carry"]:.0%}  '
      f'Lease={CAPITAL_WEIGHTS["lease"]:.0%}  '
      f'EFP={CAPITAL_WEIGHTS["efp"]:.0%}')
print(f'Vol targets: per-strategy={TARGET_STRAT_VOL:.0%}  portfolio={TARGET_PORT_VOL:.0%}')
print(f'IS: {FULL_START.date()} \u2192 {IS_END.date()}   OOS: {OOS_START.date()} \u2192 present')

NAV reference: $100M
Capital weights: TSMOM=60%  C&C=10%  Lease=20%  EFP=10%
Vol targets: per-strategy=10%  portfolio=10%
IS: 2018-01-01 → 2022-12-31   OOS: 2023-01-01 → present


In [2]:
# ── Load TSMOM ───────────────────────────────────────────────────────────
# Prefer full-history local files (BQL files may only cover partial period)
# Glob picks latest file by lexicographic sort (datestamped YYYYMMDD suffix)
tsmom_files_local = sorted(glob.glob(str(OUT_DIR / 'tsmom_portfolio_equity_local_*.csv')))
tsmom_files_bql   = sorted(glob.glob(str(OUT_DIR / 'tsmom_portfolio_equity_[0-9]*.csv')))

r_tsmom = None
if tsmom_files_local:
    path = tsmom_files_local[-1]   # latest local (full history)
    print(f'Loading TSMOM from: {path}')
    df_ts  = pd.read_csv(path, parse_dates=['date'], index_col='date')
    r_tsmom = df_ts['portfolio_ret_net'].rename('tsmom')
    r_tsmom.index = pd.to_datetime(r_tsmom.index)
elif tsmom_files_bql:
    path = tsmom_files_bql[-1]
    print(f'Loading TSMOM (BQL) from: {path}')
    df_ts  = pd.read_csv(path, parse_dates=['date'], index_col='date')
    r_tsmom = df_ts['portfolio_ret_net'].rename('tsmom')
    r_tsmom.index = pd.to_datetime(r_tsmom.index)
else:
    warnings.warn('TSMOM equity file not found \u2014 using zero returns')

if r_tsmom is not None:
    nz = (r_tsmom != 0).sum()
    print(f'  Date range  : {r_tsmom.index[0].date()} \u2192 {r_tsmom.index[-1].date()}')
    print(f'  Trading days: {nz} nonzero returns')
    print(f'  Ann return  : {r_tsmom.mean() * ANN:.2%}')

Loading TSMOM from: outputs/tsmom_portfolio_equity_local_20260307.csv
  Date range  : 2018-01-01 → 2026-02-27
  Trading days: 1847 nonzero returns
  Ann return  : 5.09%


In [3]:
# ── Load Lease Rate Curve ────────────────────────────────────────────────
# pnl_bp = daily bp P&L in raw units; vol-scaling handles normalisation
lease_path = OUT_DIR / '06_portfolio_pnl.csv'
r_lease = None
if not lease_path.exists():
    warnings.warn(f'Lease portfolio file not found: {lease_path}')
else:
    df_l    = pd.read_csv(lease_path, parse_dates=['date'], index_col='date')
    df_l.index = pd.to_datetime(df_l.index)
    r_lease = df_l['pnl_bp'].rename('lease')
    nz = (r_lease != 0).sum()
    print(f'Lease  | {r_lease.index[0].date()} \u2192 {r_lease.index[-1].date()} | '
          f'{nz} nonzero days | ann={r_lease.mean()*ANN:.2%}')

Lease  | 2018-01-01 → 2026-02-22 | 2147 nonzero days | ann=367.50%


In [4]:
# ── Load COMEX EFP ───────────────────────────────────────────────────────
# pnl_bp = daily bp P&L in raw units; vol-scaling handles normalisation
efp_path = COMEX_DIR / '06_combined_portfolio.csv'
r_efp = None
if not efp_path.exists():
    warnings.warn(f'EFP portfolio file not found: {efp_path}')
else:
    df_e  = pd.read_csv(efp_path, parse_dates=['date'], index_col='date')
    df_e.index = pd.to_datetime(df_e.index)
    r_efp = df_e['pnl_bp'].rename('efp')
    nz = (r_efp != 0).sum()
    print(f'EFP    | {r_efp.index[0].date()} \u2192 {r_efp.index[-1].date()} | '
          f'{nz} nonzero days | ann={r_efp.mean()*ANN:.2%}')

EFP    | 2018-01-01 → 2026-02-22 | 2322 nonzero days | ann=311.93%


In [5]:
# ── Load Cash & Carry (daily reconstruction from trade log) ─────────────
# totalpnlbp distributed uniformly across each trade's hold period
# This is a simplification vs exact daily carry accrual — adequate for portfolio metrics

def reconstruct_cc_daily(trades_path: Path, date_index: pd.DatetimeIndex) -> pd.Series:
    """Spread totalpnlbp uniformly across each trade's hold period -> daily bp return."""
    df = pd.read_csv(trades_path, parse_dates=['entrydate', 'exitdate'])
    pnl = pd.Series(0.0, index=date_index, dtype=float)
    for _, row in df.iterrows():
        mask = (date_index >= row['entrydate']) & (date_index <= row['exitdate'])
        n = int(mask.sum())
        if n > 0:
            pnl[mask] += row['totalpnlbp'] / n
    return pnl

# Build business-day date index
date_index = pd.bdate_range(start=FULL_START, end=pd.Timestamp.today())

gc_path = COMEX_DIR / 'cc_trades_gc.csv'
si_path = COMEX_DIR / 'cc_trades_si.csv'
r_carry = None
if gc_path.exists() and si_path.exists():
    r_gc    = reconstruct_cc_daily(gc_path, date_index)
    r_si    = reconstruct_cc_daily(si_path, date_index)
    r_carry = ((r_gc + r_si) / 2).rename('carry')   # equal-weight GC & SI
    nz = (r_carry != 0).sum()
    print(f'C&C    | {r_carry.index[0].date()} \u2192 {r_carry.index[-1].date()} | '
          f'{nz} nonzero days | ann={r_carry.mean()*ANN:.2%}')
    print(f'       | GC trades={len(pd.read_csv(gc_path))}  SI trades={len(pd.read_csv(si_path))}')
else:
    warnings.warn('C&C trade logs not found')

C&C    | 2018-01-01 → 2026-03-20 | 884 nonzero days | ann=1132.18%
       | GC trades=34  SI trades=32


In [6]:
# ── Align: build common daily return panel ───────────────────────────────
series = {k: v for k, v in {
    'tsmom': r_tsmom, 'carry': r_carry, 'lease': r_lease, 'efp': r_efp
}.items() if v is not None}

if not series:
    raise RuntimeError('No strategy data loaded. Run prerequisite notebooks first.')

rets = pd.DataFrame(series)
rets = rets.loc[FULL_START:]     # trim to common start
rets = rets.fillna(0.0)          # no-trade days = zero return
rets.index.name = 'date'

print('Data coverage summary:')
print(f'  Common date range: {rets.index[0].date()} \u2192 {rets.index[-1].date()}')
print(f'  Total rows: {len(rets)}')
print()
for col in rets.columns:
    nonzero = (rets[col] != 0).sum()
    ann_ret = rets[col].mean() * ANN
    ann_vol = rets[col].std() * ANN**0.5
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0.0
    print(f'  {col:10s}: {nonzero:5d} active days | '
          f'ann_ret={ann_ret:+.2%}  vol={ann_vol:.2%}  Sharpe={sharpe:.2f}')

missing = [k for k in ['tsmom','carry','lease','efp'] if k not in rets.columns]
if missing:
    warnings.warn(f'Missing strategies (substituting zero): {missing}')
    for k in missing:
        rets[k] = 0.0

Data coverage summary:
  Common date range: 2018-01-01 → 2026-03-20
  Total rows: 2991

  tsmom     :  1847 active days | ann_ret=+3.63%  vol=14.80%  Sharpe=0.25
  carry     :   884 active days | ann_ret=+811.94%  vol=746.46%  Sharpe=1.09
  lease     :  2147 active days | ann_ret=+365.04%  vol=261.49%  Sharpe=1.40
  efp       :  2322 active days | ann_ret=+309.85%  vol=249.94%  Sharpe=1.24


In [7]:
def ewma_vol(r: pd.Series, lam: float = EWMA_LAMBDA, ann: int = ANN,
             target_vol: float = TARGET_STRAT_VOL) -> pd.Series:
    """One-period lagged EWMA annualised vol (avoids lookahead bias).
    Used for the portfolio-level overlay only.
    alpha = 1 - lam = 0.06 (94% persistence).
    """
    var = r.ewm(com=lam / (1 - lam), min_periods=20).var()
    return (var * ann).pow(0.5).shift(1).fillna(target_vol)


# ── Vol scaling: per-strategy constant vol target ────────────────────────
# Constant scaling (vs EWMA) robustly handles:
#   • sparse series (C&C: only ~30% active days)
#   • non-decimal units (Lease/EFP raw bp, TSMOM decimal)
# Sharpe ratios are exactly preserved under any linear scaling.

scaled             = {}
strat_scales_const = {}

for strat in ['tsmom', 'carry', 'lease', 'efp']:
    r = rets[strat]
    raw_ann_vol = r.std() * ANN**0.5
    if raw_ann_vol < 1e-12:
        factor = 0.0
    else:
        factor = min(TARGET_STRAT_VOL / raw_ann_vol, LEV_CAP)
    strat_scales_const[strat] = factor
    scaled[strat] = r * factor

# For gross-leverage computation downstream, wrap in constant Series
strat_scales = {
    s: pd.Series(strat_scales_const[s], index=rets.index)
    for s in strat_scales_const
}

rets_scaled = pd.DataFrame(scaled)

print('Per-strategy vol scaling (constant factor):')
hdr = f'  {"":<10s}  {"scale":>7s}  {"raw_vol":>8s}  {"real_vol":>9s}  {"sharpe":>7s}'
print(hdr)
print('-' * 48)
for s in rets_scaled.columns:
    fc  = strat_scales_const[s]
    rv  = rets_scaled[s].std() * ANN**0.5
    raw = rets[s].std() * ANN**0.5
    sr  = (rets_scaled[s].mean() * ANN) / rv if rv > 0 else 0
    print(f'  {s:<10s}  {fc:>6.3f}x  {raw:>8.4f}  {rv:>8.2%}  {sr:>7.2f}')


Per-strategy vol scaling (constant factor):
                scale   raw_vol   real_vol   sharpe
------------------------------------------------
  tsmom        0.676x    0.1480    10.00%     0.25
  carry        0.013x    7.4646    10.00%     1.09
  lease        0.038x    2.6149    10.00%     1.40
  efp          0.040x    2.4994    10.00%     1.24


In [8]:
# ── Portfolio: combine + NAV-level vol overlay ───────────────────────────

# Capital-weighted sum
port_ret = sum(CAPITAL_WEIGHTS.get(s, 0) * rets_scaled[s] for s in rets_scaled)

# Portfolio-level vol overlay
port_vol   = ewma_vol(port_ret)
port_scale = (TARGET_PORT_VOL / port_vol).clip(upper=PORT_LEV_CAP).fillna(1.0)
port_ret_f = (port_ret * port_scale).rename('portfolio')

# NAV equity curve (compound daily)
nav = NAV_USD * (1 + port_ret_f).cumprod()
nav_df = pd.DataFrame({
    'nav_usd': nav.values,
    'ret_net': port_ret_f.values,
}, index=nav.index)
nav_df.index.name = 'date'

# Gross leverage = weighted sum of per-strat scales times portfolio scale
gross_lev = sum(
    CAPITAL_WEIGHTS.get(s, 0) * strat_scales[s] for s in strat_scales
) * port_scale

avg_lev = float(gross_lev.mean())
max_lev = float(gross_lev.max())
print(f'Portfolio gross leverage: avg={avg_lev:.2f}x  max={max_lev:.2f}x')
print(f'Portfolio vol (realized): {port_ret_f.std()*ANN**0.5:.2%}')

Portfolio gross leverage: avg=0.75x  max=0.84x
Portfolio vol (realized): 8.84%


In [9]:
# ── Metrics: full / IS / OOS ─────────────────────────────────────────────

def compute_metrics(r: pd.Series) -> dict:
    """Compute standard backtest metrics from a daily return series."""
    r = r.dropna()
    if len(r) < 20:
        return {}
    ann_ret  = float(r.mean() * ANN)
    ann_vol  = float(r.std() * ANN**0.5)
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else 0.0
    cum      = (1 + r).cumprod()
    roll_max = cum.cummax()
    dd       = cum / roll_max - 1
    max_dd   = float(dd.min())
    calmar   = ann_ret / abs(max_dd) if max_dd != 0 else 0.0
    hit_rate = float((r > 0).mean())
    win_loss = float(r[r > 0].mean() / abs(r[r < 0].mean())) if (r < 0).any() else np.nan
    return dict(ann_return=ann_ret, ann_vol=ann_vol, sharpe=sharpe,
                max_dd=max_dd, calmar=calmar, hit_rate=hit_rate, win_loss=win_loss)

slices = {
    'Full': port_ret_f,
    'IS':   port_ret_f.loc[:IS_END],
    'OOS':  port_ret_f.loc[OOS_START:],
}
perf = {period: compute_metrics(r) for period, r in slices.items()}
perf_df = pd.DataFrame(perf).T

# Per-strategy attribution
strat_contrib = {}
for s in ['tsmom', 'carry', 'lease', 'efp']:
    r_scaled  = rets_scaled[s]
    r_contrib = r_scaled * CAPITAL_WEIGHTS.get(s, 0) * port_scale
    strat_contrib[s] = {
        'capital_weight': CAPITAL_WEIGHTS.get(s, 0),
        'risk_budget':    RISK_BUDGETS.get(s, 0),
        **compute_metrics(r_scaled),
        'port_contribution_ann': float(r_contrib.mean() * ANN),
    }
attr_df = pd.DataFrame(strat_contrib).T

# Stress period analysis
stress_results = {}
for label, (start, end) in STRESS_PERIODS.items():
    slc = port_ret_f.loc[start:end]
    if len(slc) >= 5:
        stress_results[label] = compute_metrics(slc)
stress_df = pd.DataFrame(stress_results).T

print(f'IS period  : {slices["IS"].index[0].date()} \u2192 {slices["IS"].index[-1].date()} ({len(slices["IS"])} days)')
print(f'OOS period : {slices["OOS"].index[0].date()} \u2192 {slices["OOS"].index[-1].date()} ({len(slices["OOS"])} days)')

IS period  : 2018-01-01 → 2022-12-30 (1825 days)
OOS period : 2023-01-01 → 2026-03-20 (1166 days)


In [10]:
# ── Summary table ────────────────────────────────────────────────────────
print('\n' + '='*62)
print(f'  MASTER PORTFOLIO \u2014 ${NAV_USD/1e6:.0f}M NAV REFERENCE BOOK')
print('='*62)
fmt_hdr = '  {:<25s}  {:>10s}  {:>10s}  {:>10s}'
print(fmt_hdr.format('Metric', 'Full', 'IS', 'OOS'))
print('-'*62)
fields = [
    ('Ann. Return (net)',  'ann_return', '{:+.1%}'),
    ('Ann. Volatility',    'ann_vol',    '{:.1%}'),
    ('Sharpe Ratio',       'sharpe',     '{:.2f}'),
    ('Max Drawdown',       'max_dd',     '{:.1%}'),
    ('Calmar Ratio',       'calmar',     '{:.2f}'),
    ('Hit Rate (daily)',   'hit_rate',   '{:.1%}'),
    ('Win/Loss Ratio',     'win_loss',   '{:.2f}'),
]
for label, key, fmt_str in fields:
    row = '  {:<25s}'.format(label)
    for period in ['Full', 'IS', 'OOS']:
        v = perf[period].get(key, float('nan'))
        row += '  {:>10s}'.format(fmt_str.format(v) if not np.isnan(v) else 'n/a')
    print(row)
print('-'*62)
avg_lev = float(gross_lev.mean())
max_lev = float(gross_lev.max())
print(f'  {"Avg Gross Leverage":<25s}  {avg_lev:>10.2f}x')
print(f'  {"Max Gross Leverage":<25s}  {max_lev:>10.2f}x')
print('='*62)

print('\nPer-strategy metrics (vol-scaled, full period):')
display(attr_df[['capital_weight','ann_return','ann_vol','sharpe','max_dd','port_contribution_ann']].round(4))

if not stress_df.empty:
    print('\nStress period analysis:')
    display(stress_df[['ann_return','sharpe','max_dd']].round(4))


  MASTER PORTFOLIO — $100M NAV REFERENCE BOOK
  Metric                           Full          IS         OOS
--------------------------------------------------------------
  Ann. Return (net)              +11.6%       +6.9%      +19.1%
  Ann. Volatility                  8.8%        8.7%        9.0%
  Sharpe Ratio                     1.32        0.79        2.12
  Max Drawdown                   -17.0%      -17.0%      -11.5%
  Calmar Ratio                     0.68        0.41        1.66
  Hit Rate (daily)                42.6%       40.9%       45.4%
  Win/Loss Ratio                   1.53        1.40        1.69
--------------------------------------------------------------
  Avg Gross Leverage               0.75x
  Max Gross Leverage               0.84x

Per-strategy metrics (vol-scaled, full period):


,capital_weight,ann_return,ann_vol,sharpe,max_dd,port_contribution_ann
tsmom,0.6,0.0245,0.1,0.2450,-0.4090,0.0372
carry,0.1,0.1088,0.1,1.0877,-0.4118,0.0095
lease,0.2,0.1396,0.1,1.3960,-0.1322,0.0482
efp,0.1,0.1240,0.1,1.2397,-0.1953,0.0215



Stress period analysis:


,ann_return,sharpe,max_dd
COVID (2020-03),0.1126,1.0658,-0.1064
Russia/Ukraine (2022),0.1934,1.4521,-0.0375
SVB (2023-03),0.3064,3.6235,-0.0283
Tariff Shock (2025),0.2830,3.5917,-0.0315


In [11]:
# ── Chart: NAV equity curve + drawdown ───────────────────────────────────

cum       = (1 + port_ret_f).cumprod()
roll_max  = cum.cummax()
dd_series = cum / roll_max - 1
nav_m     = nav / 1e6   # in $M

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=['NAV ($M)', 'Drawdown (%)'],
                    row_heights=[0.65, 0.35])

# NAV curve
fig.add_trace(go.Scatter(x=nav_m.index, y=nav_m.values,
                         name='NAV', line=dict(color='royalblue', width=2)), row=1, col=1)

# IS/OOS boundary — use add_shape to avoid Plotly annotation bug with string datetimes
_is_end_str = str(IS_END.date())
for r in [1, 2]:
    fig.add_shape(type='line', x0=_is_end_str, x1=_is_end_str, y0=0, y1=1,
                  xref='x', yref='y domain', line=dict(dash='dash', color='grey'),
                  row=r, col=1)
# Annotation only on row 1
fig.add_annotation(x=_is_end_str, xref='x', y=1.02, yref='y domain',
                   text='IS/OOS', showarrow=False, font=dict(size=9, color='grey'),
                   row=1, col=1)

# Stress period shading
colors = px.colors.qualitative.Set2
for ci, (label, (s, e)) in enumerate(STRESS_PERIODS.items()):
    col = colors[ci % len(colors)]
    for r in [1, 2]:
        fig.add_vrect(x0=s, x1=e, fillcolor=col, opacity=0.12,
                      layer='below', line_width=0, row=r, col=1)

# Drawdown
fig.add_trace(go.Scatter(x=dd_series.index, y=dd_series.values * 100,
                         name='Drawdown', fill='tozeroy',
                         line=dict(color='tomato', width=1)), row=2, col=1)
# Threshold lines on drawdown
fig.add_hline(y=-PAUSE_DD_THRESHOLD * 100, line_dash='dash', line_color='orange',
              annotation_text=f'Pause ({PAUSE_DD_THRESHOLD:.0%})', row=2, col=1)
fig.add_hline(y=-HARD_CAP_DD * 100, line_dash='dash', line_color='red',
              annotation_text=f'Hard cap ({HARD_CAP_DD:.0%})', row=2, col=1)

# Sharpe annotations
sharpe_is  = perf.get('IS',  {}).get('sharpe', float('nan'))
sharpe_oos = perf.get('OOS', {}).get('sharpe', float('nan'))
fig.add_annotation(x=str(pd.Timestamp('2021-06-01').date()),
                   y=nav_m.max() * 0.92, text=f'IS Sharpe: {sharpe_is:.2f}',
                   showarrow=False, font=dict(size=12), row=1, col=1)
fig.add_annotation(x=str(pd.Timestamp('2024-06-01').date()),
                   y=nav_m.max() * 0.92, text=f'OOS Sharpe: {sharpe_oos:.2f}',
                   showarrow=False, font=dict(size=12), row=1, col=1)

fig.update_layout(
    title=f'Master Portfolio \u2014 ${NAV_USD/1e6:.0f}M NAV',
    height=600, template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.update_yaxes(title_text='$M', row=1, col=1)
fig.update_yaxes(title_text='%', row=2, col=1)
fig.show()

In [12]:
# ── Chart: per-strategy cumulative returns (vol-scaled) ──────────────────

strat_order = ['tsmom', 'carry', 'lease', 'efp']
strat_names = {'tsmom': 'TSMOM', 'carry': 'Cash & Carry', 'lease': 'Lease Rate', 'efp': 'COMEX EFP'}
strat_colors = {'tsmom': 'steelblue', 'carry': 'seagreen',
                'lease': 'darkorange', 'efp': 'mediumpurple'}

fig2 = make_subplots(rows=2, cols=2, shared_xaxes=True,
                     subplot_titles=[strat_names[s] for s in strat_order],
                     vertical_spacing=0.12, horizontal_spacing=0.08)
positions = [(1,1),(1,2),(2,1),(2,2)]

for strat, (row, col) in zip(strat_order, positions):
    r = rets_scaled[strat]
    cum_s = (1 + r).cumprod()
    m = compute_metrics(r)
    title = (f'{strat_names[strat]}  |  w={CAPITAL_WEIGHTS[strat]:.0%}  '
             f'Sharpe={m.get("sharpe",0):.2f}  MDD={m.get("max_dd",0):.1%}')
    fig2.layout.annotations[positions.index((row,col))].text = title
    fig2.add_trace(
        go.Scatter(x=cum_s.index, y=cum_s.values,
                   name=strat_names[strat],
                   line=dict(color=strat_colors[strat], width=1.5)),
        row=row, col=col
    )
    fig2.add_hline(y=1.0, line_dash='dot', line_color='grey',
                   line_width=1, row=row, col=col)

fig2.update_layout(title='Per-strategy cumulative returns (vol-scaled to 10%)',
                   height=600, template='plotly_white', showlegend=False)
fig2.show()

In [13]:
# ── Chart: rolling Sharpe / vol / gross leverage ──────────────────────────

roll_win = 252

roll_sharpe = (port_ret_f.rolling(roll_win).mean() * ANN /
               (port_ret_f.rolling(roll_win).std() * ANN**0.5)).rename('sharpe')
roll_vol    = (port_ret_f.rolling(roll_win).std() * ANN**0.5 * 100).rename('vol_pct')

fig3 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                     subplot_titles=['Rolling 252d Sharpe',
                                     'Rolling 252d Volatility (%)',
                                     'Gross Leverage (x)'])

fig3.add_trace(go.Scatter(x=roll_sharpe.index, y=roll_sharpe.values,
                          name='Rolling Sharpe', line=dict(color='royalblue')),
               row=1, col=1)
fig3.add_hline(y=0, line_dash='dot', line_color='grey', row=1, col=1)
fig3.add_hline(y=1.0, line_dash='dash', line_color='green',
               annotation_text='Target Sharpe=1', row=1, col=1)

fig3.add_trace(go.Scatter(x=roll_vol.index, y=roll_vol.values,
                          name='Rolling Vol', line=dict(color='darkorange')),
               row=2, col=1)
fig3.add_hline(y=TARGET_PORT_VOL*100, line_dash='dash', line_color='grey',
               annotation_text=f'Target {TARGET_PORT_VOL:.0%}', row=2, col=1)

fig3.add_trace(go.Scatter(x=gross_lev.index, y=gross_lev.values,
                          name='Gross Leverage', line=dict(color='mediumpurple')),
               row=3, col=1)
fig3.add_hline(y=PORT_LEV_CAP, line_dash='dash', line_color='red',
               annotation_text=f'Cap {PORT_LEV_CAP:.1f}x', row=3, col=1)

fig3.update_layout(title='Rolling risk metrics and portfolio leverage',
                   height=700, template='plotly_white', showlegend=False)
fig3.show()

In [14]:
# ── Export: save all summary tables ──────────────────────────────────────
perf_df.to_csv(MB_DIR / 'master_summary.csv')
nav_df.to_csv(MB_DIR / 'master_nav.csv')
attr_df.to_csv(MB_DIR / 'master_attribution.csv')

if not stress_df.empty:
    stress_df.to_csv(MB_DIR / 'master_stress.csv')

gross_lev.rename('gross_leverage').to_frame().to_csv(MB_DIR / 'master_leverage.csv')

# Strategy-level scaled returns for further analysis
rets_scaled.to_csv(MB_DIR / 'master_strategy_returns.csv')

print(f'Outputs written to {MB_DIR}/')
for f in sorted(MB_DIR.glob('*.csv')):
    print(f'  {f.name}')

Outputs written to outputs_masterbacktest/
  master_attribution.csv
  master_leverage.csv
  master_nav.csv
  master_strat_returns.csv
  master_strategy_returns.csv
  master_stress.csv
  master_summary.csv
